# Lesson 4.4: Collaborative Location Review


**🎬 Video:** [Lesson 4.4: Collaborative Location Review](#)

## Overview

The geoparser resolved hundreds of place names from Reddit posts, but automatic geocoders make predictable mistakes:

| Error type | Example |
|---|---|
| Wrong disambiguation | `JMU` → Jiamusi, China instead of James Madison University |
| Wrong coordinates | `Rockingham County` placed in the New Hampshire and not Virginia |
| False positive | `Virginia` captured as a destination when it is just context |
| Spurious match | `D-Hall` → a mall in New York |

Your team will work through these locations together in Google Sheets, then export a clean file for Lessons 5 and 6.

**Workflow overview**

| Part | Tool | Who |
|---|---|---|
| 1 — Load the Review File | Python (this notebook) | One student |
| 2 — Setup Google Sheet | Google Sheets | One student |
| 3 — Publish Google Sheet | Google Sheets | One student |
| 4 — Branch, Merge, Sync | Codespaces | One student |
| 5 & 6 — Recursive Review | Python (this notebook)/Google Sheets | Whole Team |
| 7 — Branch and Export the Cleaned file | Python (this notebook) | One student |
| 8 — Check progress | Python (this notebook) | Whole Team |


---

## 1 Load the Review File

Run the cell below to confirm the file is ready. You will see columns the `geoparser` created. These include information about the location it retrieved such as the state name (admin_1_name) and the country name. These columns can help you figure out quickly if the geoparser was relatively close to where it needed to be for that location. The second additional set of columns will help with making corrections to the existing data. These include:
- `action` 
- `corrected_name`
- `corrected_latlon`
- `corrected_place_type`
- `reviewer`
- `place_count`


In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/JMU/JMU_geoparsed_long.csv")

df.iloc[: , 5:20].head()


,place,latitude,longitude,feature_type,admin1_name,admin2_name,country_name,school,place_type,action,corrected_name,corrected_latlon,corrected_place_type,reviewer,place_count
0,City of Harrisonburg,38.44957,-78.86892,second-order administrative division,Virginia,City of Harrisonburg,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,87
1,City of Harrisonburg,38.44957,-78.86892,second-order administrative division,Virginia,City of Harrisonburg,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,87
2,City of Harrisonburg,38.44957,-78.86892,second-order administrative division,Virginia,City of Harrisonburg,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,87
3,City of Harrisonburg,38.44957,-78.86892,second-order administrative division,Virginia,City of Harrisonburg,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,87
4,City of Harrisonburg,38.44957,-78.86892,second-order administrative division,Virginia,City of Harrisonburg,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,87


---

## 2 Setup Google Sheet (One person: Geodata Manager)

This step should be done by one person only. This person will invite their other group members as the last step.


### 2.1 Import the file

1. Download the `JMU_geoparsed_long.csv` file to a directory on our local machine (laptop, desktop)
   1. Find the file in `data/JMU`
   2. <img src="../lesson_assets/images/windows.svg" alt="Windows" width="14"> `Right-Click`| <img src="../lesson_assets/images/apple.svg" alt="Mac" width="14"> `Cmd + Click`
   3. Select `Download...` from the menu.
      1. <img src="../lesson_assets/images/download-csv-file.png" alt="download csv file" width="400">
   4. Save the file to an easy to locate directory like Desktop or Downloads
2. Go to [sheets.google.com](https://sheets.google.com) → **Blank spreadsheet**
   1. <img src="../lesson_assets/images/gsheet-blank-spreadsheet.png" alt="Create a blank spreadsheet" width="400">
3. **File → Import → Upload** → **Browse**
   1. select `JMU_geoparsed_long.csv` from your local directory.
   2. <img src="../lesson_assets/images/gsheet-file-import.png" alt="file import" width="400">
   3. <img src="../lesson_assets/images/gsheet-upload-browse.png" alt="upload browse" width="400">
4. Choose **Replace spreadsheet**, separator type **Comma**, click **Import**.
   1. <img src="../lesson_assets/images/gsheet-import-file.png" alt="File import settings" width="400">
5. Rename the spreadsheet: `JMU_geoparsed_long_cleaned`
   1. <img src="../lesson_assets/images/gsheet-rename.png" alt="Rename file" width="400">





### 2.2 Add data validation

Set up three dropdown validations so the whole team fills in consistent values.

**Column `action`** *(most important)*:

1. Click the `action` column header to select the whole column
2. **Data → Data validation**
   1. <img src="../lesson_assets/images/gsheet-data-validation.png" alt="data validation" width="400">
3. **Add Rule**
   1. <img src="../lesson_assets/images/gsheet-add-rule.png" alt="Add rule" width="100">  
4. Set range to start from cell `O2` and not `O1`.
   1. This prevents the column header `action` from being included in the data validation rule.
   2. Leave criteria to **Dropdown**
   3. Click **Done**
   4. <img src="../lesson_assets/images/gsheet-data-validation-setup.png" alt="data validation setup" width="400">
5.  Add three options: `KEEP`, `CORRECT`, `REMOVE`
    1.  Set a color for each as a visual cue.
    2.  <img src="../lesson_assets/images/gsheet-keep-correct-remove.png" alt="" width="400">
6. <img src="../lesson_assets/images/windows.svg" alt="Windows" width="14"> `Right-Click`| <img src="../lesson_assets/images/apple.svg" alt="Mac" width="14"> `Cmd + Click` the `action` header cell → **Insert note**
   1. <img src="../lesson_assets/images/gsheet-insert-note.png" alt="Insert note" width="400"> 
7. Copy+Paste: *KEEP = location is correct. CORRECT = right place, wrong details. REMOVE = not a real location or geoparser error.*
   1. <img src="../lesson_assets/images/gsheet-note-text.png" alt="note text" width="400">

**Column `reviewer`**:

1. Select the `reviewer` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add each team member's name

**Column `corrected_place_type`**:

1. Select the `corrected_place_type` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add: 
- `Country`
- `State`
- `Region`
- `City`
- `Neighborhood`
- `University`
- `Road`
- `Building`
- `Natural Feature`



### 2.3 Filter and Sort

You want to prioritize verifying and corrected locations that appear often and work your way down. Ideally, you would correct all locations for which there are at least three occurrences.

1. Select the header row by clicking on the row number for row 1
2. Click **Filter** on the toolbar
   1. <img src="../lesson_assets/images/gsheet-create-filter.png" alt="Create filter" width="400">
3. Click the drop-down triangle next to `place_count`.
   1. Click **Sort Z to A**
   2. <img src="../lesson_assets/images/gsheet-sort.png" alt="Sort" width="400">
4. Harrisonburg should now top the list with the most entries


### 2.3 Invite Collaborators

Once the sheet is properly setup, invite your team. 

1. **Share → Anyone with the link → Editor** → copy the link and share it with your team
   1. If you have people's gmail accounts, you can also invite them directly
   2. <img src="../lesson_assets/images/gsheet-share.png" alt="Share" width="100">

---
## 3 Publish the Google Sheet

With this basic setup complete, set your file to go live on the web. Be aware that it will be viewable by anyone. It is good practice to unpublish it at the end of this project. 

1. **File → Share → Publish to web**
   1. <img src="../lesson_assets/images/gsheet-publish-to-web.png" alt="Publish to web" width="400">
2. Choose: **Entire document** → **Comma-separated values (.csv)**
   1. <img src="../lesson_assets/images/gsheet-publish-settings.png" alt="Publish settings" width="400">
3. Click **Ok** if asked if you are sure you want to publish
4. Click **Publish** → copy the URL to your clipboard. You will use it in the next section.
   1. <img src="../lesson_assets/images/gsheet-share-link.png" alt="Publish settings" width="400">
   
   


---
## 4 Branch, Merge, and Sync

1. Create a new branch from `main`
2. Paste the URL into the `SHEETS_LINK` variable in **Part 6** below
3. Run the cells to verify that the locations show on the map
   1. You should get the following message: ✅ No validation issues found.
4. Once everything runs, merge your branch back to `main`
   1. Remember the full process is:
      1. > `main` → pull → create branch → edit → commit → publish branch → open PR → merge PR → return to `main` → pull
5. All other team members should pull down the changes once the PR is complete


---
## 5 Recursive Review

### Overview

The code below fetches the data from the published Google sheet and generates a map with three location types:

- Unverified
- Correct
- Keep

All `REMOVE` locations will not appear. 

Occasionally, the here can be a delay in the sheet refresh. If that happens, wait a moment and run the code again. 

Clean up the entries in the Google Sheet till you have a good data set to work with.

In the final step **one** team member should create a new branch, export the data, commit, and merge back into `main`.

### 5.1 Workflow

Work through the rows as a team. For each location:

- Read the `sentences` column to understand the context
- Check `place`, `latitude`, `longitude`, and `place_type`
- Set `action` to `KEEP`, `CORRECT`, or `REMOVE`
- If `CORRECT`: fill in only the columns that need changing:
  - `corrected_name` — new place name
  - `corrected_latlon` — paste directly from Google Maps (e.g. `38.433998, -78.872973`)
  - `corrected_place_type` — select from the dropdown
- Enter your name in `reviewer`

#### 5.1.1 Getting coordinates from Google Maps

1. Make sure that you use the site [maps.google.com](https://www.google.com/maps)
   1. **DO NOT** use Google search to find a location and click the resulting map. This is a stripped down version of Google maps that won't allow you to extract coordinates.
2. Search for the location in Google maps
3. Determine whether this is the right location
4. If a pin (<img src="../lesson_assets/images/gmaps-pin.png" alt="Publish settings" width="14">) appears for the search result: 
   1. <img src="../lesson_assets/images/windows.svg" alt="Windows" width="14"> `Right-Click`| <img src="../lesson_assets/images/apple.svg" alt="Mac" width="14"> `Cmd + Click` the pin → click the coordinates at the top of the menu → they copy automatically.
   2.  <img src="../lesson_assets/images/gmaps-save-location.png" alt="Save coordinates" width="200">)
5. If **no** Pin (<img src="../lesson_assets/images/gmaps-pin.png" alt="Publish settings" width="14">) appears for the location (i.e. a specific area on the Quad).
   1. <img src="../lesson_assets/images/windows.svg" alt="Windows" width="14"> `Right-Click`| <img src="../lesson_assets/images/apple.svg" alt="Mac" width="14"> `Cmd + Click` an area on the map that is close to where you think the location is → click the coordinates at the top of the menu → they copy automatically.
   2. <img src="../lesson_assets/images/gmaps-no-pin-coordinates.png" alt="Save coordinates" width="200">)
6. Paste the full string (`38.433998, -78.872973`) into `corrected_latlon`
   1. Python will automatically turn this into separate latitude and longitude coordinates on import.

### 5.2 Errors

Some basic error handling has been written into the code. If you encounter any of these errors, consult your most recent changes and see how you can fix them:

⚠️  *n* row(s) have unrecognised 'action' values - The 'action' columns contains a value that is not KEEP, CORRECT, or REMOVE

⚠️  *n* row(s) marked CORRECT have no corrected coordinates or name - You entered CORRECT in the spreadsheet, but then did not enter any subsequent values.

⚠️  *n* row(s) with 'corrected_latlon' value with no comma - Python is expecting a comma between lat and long in `corrected_latlon`. This usually happens if you accidentally edit a cell.

⚠️  *n* row(s) have a non-numeric corrected latitude or longitude - There are numbers or special characters in your corrected latlon values. This usually happens if you accidentally edit a `corrected_latlon` cell.

### 5.3 Merging duplicate locations

Harrisonburg appears twice in the data. First, as Harrisonburg and second as City of Harrisonburg. You want to CORRECT *both* entries. This is too ensure that they are both set to the same coordinates. You could have a scenario where one Harrisonburg has the coordinates (38.44957, -78.86892) and the second Harrisonburg from Google Maps has the coordinates (38.447155573199026, -78.8666273603042). These are virtually similar, but different enough that they appear as two separate locations. For the sake of consistency, use the Google Maps coordinates.

### 5.4 Tips

#### Work from high location count to low count
The sheet should be sorted by location count. Start with high numbers will eliminate a lot of rows on teh spreadsheet right off the bat.

#### Work in batches
If you know that all the sentences for one location belong in that location you can **copy + paste** the values all the way down for that location.

#### Filter Out Verified Entries
Filtering out verified entries as you go can help reduce the amount of scrolling you have to do.

1. Click the green drop-down arrow next to the `action` column
2. Uncheck the actions you don't want to view (i.e. CORRECT, KEEP) and Click **OK**.
   1. <img src="../lesson_assets/images/gsheet-filter-completed.png" alt="Filter values" width="300">
3. The published link always accesses the full CSV file, so there is no danger that the filter removes data.
   
#### Buildings

If a building is part of a university, place the abbreviation for the university in parentheses after the building name to help the user understand what the building is: i.e. D-Hall (JMU)


---
## 6 Data and Map

In [7]:
import sys; sys.path.insert(0, "../tests")
from helpers import load_and_validate_review_sheet

# ──────────────────────────────────────────────────────────────────────────────
# ⚠️  BEFORE RUNNING THIS CELL:
#   1. Complete Part 2
#   2. File → Share → Publish to web → Entire document → CSV → Publish
#   3. Copy the published URL and paste it below, replacing the placeholder
# ──────────────────────────────────────────────────────────────────────────────

SHEETS_LINK = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQIRZwfvrTFs2oz0lDjN2mOkeqdlnVrzsi1_QxW7-OlcJW2FxaoT7m93NeTMf1WdnVNaYnoIjJNCoS7/pub?output=csv"

if "PASTE_YOUR_LINK_HERE" in SHEETS_LINK:
    print("❌ Update SHEETS_LINK before running this cell.")
    print("   File → Share → Publish to web → Entire document → CSV → Publish → copy URL.")
else:
    df_review = load_and_validate_review_sheet(SHEETS_LINK)

✅ Loaded 884 rows from Google Sheets

Review progress:
action
NaN    884

0 / 884 rows reviewed (0%)

── Validation warnings ──────────────────────────────────
⚠️  884 row(s) have a non-numeric corrected lat:
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Fix these in the sheet, then re-run this cell before exporting.


In [10]:
# Verification map — inspect dots geographically
# Points far outside Virginia / the US East Coast are worth checking

df_review = load_and_validate_review_sheet(SHEETS_LINK)

if "df_review" not in dir():
    print("❌ df_review is not loaded yet.")
    print("   Run the cell above first (paste your SHEETS_LINK and run it), then re-run this cell.")
else:
    df_map = df_review[
        df_review["action"].isin(["KEEP", "CORRECT"]) |
        df_review["action"].isna() |
        (df_review["action"] == "")
    ].copy()

    df_map["action"] = df_map["action"].fillna("UNVERIFIED")

    df_map["lat_plot"] = pd.to_numeric(
        df_map["corrected_lat"].where(
            df_map["corrected_lat"].notna() & (df_map["corrected_lat"].astype(str).str.strip() != ""),
            df_map["latitude"]
        ), errors="coerce")

    df_map["lon_plot"] = pd.to_numeric(
        df_map["corrected_lon"].where(
            df_map["corrected_lon"].notna() & (df_map["corrected_lon"].astype(str).str.strip() != ""),
            df_map["longitude"]
        ), errors="coerce")

    df_map["place"] = df_map["corrected_name"].where(
        df_map["corrected_name"].notna() & (df_map["corrected_name"] !=""), df_map["place"])

    df_map["corrected_place_type"] = df_map["corrected_place_type"].where((df_map["action"]=="CORRECT") &
               df_map["corrected_place_type"].notna() & (df_map["corrected_place_type"] !=""), "None")

    df_map = df_map.dropna(subset=["lat_plot", "lon_plot"])

    df_map = (
        df_map.groupby(["place", "action"], sort=False)
        .agg(
            corrected_name=("corrected_name", "first"),
            location_count=("place", "size"),
            lat_plot=("lat_plot", "first"),
            lon_plot=("lon_plot", "first"),
            place_type=("place_type", "first"),
            corrected_place_type=("corrected_place_type", "first"),
            reviewer=("reviewer", "first"),
        )
        .reset_index()
    )

    df_map[["corrected_place_type", "reviewer", "corrected_name"]] = df_map[["corrected_place_type", "reviewer","corrected_name"]].fillna("None")

    fig = px.scatter_map(
        df_map,
        lat="lat_plot", lon="lon_plot",
        size="location_count",
        hover_name="place",
        labels={"lat_plot":"latitude","lon_plot":"longitude"},
        hover_data={
            "corrected_name": True,
            "lat_plot": True,
            "lon_plot": True,
            "place_type": True,
            "corrected_place_type": True,
            "action": True,
            "reviewer": True
        },
        color="action",
        color_discrete_map={
            "KEEP": "#2ca02c",
            "CORRECT": "#ff7f0e",
            "UNVERIFIED": "#aec7e8",
        },
       
        size_max=25,
        map_style="carto-positron",
        center={
            "lat": 37.5,
            "lon": -78.0,
        },
        zoom=4,
        height=500,
        title="Location review map — hover for details"
    )
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        }
    )
    fig.update_traces(marker=dict(sizemin=3))

    fig.show()

    print("\n💡 Dots far outside Virginia may be geoparser errors.")    

✅ Loaded 884 rows from Google Sheets

Review progress:
action
NaN        676
CORRECT    131
KEEP        77

208 / 884 rows reviewed (24%)

── Validation warnings ──────────────────────────────────
⚠️  753 row(s) have a non-numeric corrected lat:
87    NaN
88    NaN
89    NaN
90    NaN
91    NaN
⚠️  753 row(s) have a non-numeric corrected lon:
87    NaN
88    NaN
89    NaN
90    NaN
91    NaN
Fix these in the sheet, then re-run this cell before exporting.



💡 Dots far outside Virginia may be geoparser errors.


---

## 7 Branch and Export the Cleaned File

When your team is satisfied with the review you will create an export copy. This process is involved, so make sure you are in a good place. This process will:

- Drop every row marked `REMOVE`
- Apply any corrections (`corrected_name` → `place`, `corrected_latlon` → `latitude` / `longitude`, etc.)
- Remove the six review columns
- Save the result as `../data/JMU/JMU_geoparsed_cleaned.csv`

This file feeds directly into Lessons 5 and 6, and if you did a good job cleaning the data at this point you will get better results down the road.


**One student on the team** does these steps after cleaning is complete.

This is the same branch → commit → pull request workflow from [Lesson 1.1](../lesson_1_the_team/lesson_1_1_git_and_pull_requests.ipynb). Refer back to that lesson if you need a refresher.

1. Create a new branch named `location-review` from the status bar
2. Run export cell below
3. Open **Source Control** and stage `data/JMU/JMU_geoparsed_long.csv` and `data/JMU/JMU_geoparsed_cleaned.csv`
4. Commit with a message like `review: update JMU location data`, then **Commit & Sync**
5. Publish the branch and open a Pull Request from `location-review` → `main`
6. Merge the PR on GitHub, delete the branch, switch back to `main`, and sync

> 👉 **Note:** *Do not commit any other files. If unexpected files appear under Changes, discard them before committing.*




In [12]:
# Apply corrections and export the cleaned file

if "df_review" not in dir():
    print("❌ df_review is not loaded yet.")
    print("   Run the Part 6 cell first (paste your SHEETS_LINK and run it), then re-run this cell.")
else:
    df_out = df_review.copy()

    long_path  = "../data/JMU/JMU_geoparsed_long.csv"
    clean_path = "../data/JMU/JMU_geoparsed_cleaned.csv"

    # Drop rows marked REMOVE
    n_before = len(df_out)
    df_out = df_out[df_out["action"] != "REMOVE"].copy()
    n_removed = n_before - len(df_out)

    # Apply corrections where non-empty
    def apply_correction(df, corrected_col, target_col, cast=None):
        mask = df[corrected_col].notna() & (df[corrected_col].astype(str).str.strip() != "")
        if cast:
            df.loc[mask, target_col] = pd.to_numeric(df.loc[mask, corrected_col], errors="coerce")
        else:
            df.loc[mask, target_col] = df.loc[mask, corrected_col]
        return df, mask.sum()

    df_out, n_name = apply_correction(df_out, "corrected_name",       "place")
    df_out, n_lat  = apply_correction(df_out, "corrected_lat",        "latitude",  cast=True)
    df_out, n_lon  = apply_correction(df_out, "corrected_lon",        "longitude", cast=True)
    df_out, n_type = apply_correction(df_out, "corrected_place_type", "place_type")

    # ── Write reviewed state back to long CSV (preserves action columns for grading) ──
    # Drop the split helper columns (corrected_lat/lon) — they're derived from corrected_latlon
    keep_cols = [c for c in df_review.columns if c not in ["corrected_lat", "corrected_lon"]]
    df_review[keep_cols].to_csv(long_path, index=False)
    print(f"✅ Updated  {long_path}")

    # ── Save cleaned file ──────────────────────────────────────────────────────────
    review_cols = ["action", "corrected_name", "corrected_latlon", "corrected_lat", "corrected_lon",
                   "corrected_place_type", "reviewer", "place_count"]
    df_out = df_out.drop(columns=[c for c in review_cols if c in df_out.columns])
    df_out.to_csv(clean_path, index=False)

    print(f"✅ Saved    {clean_path}")
    print(f"\nSummary:")
    print(f"  {n_before:,} rows in  →  {len(df_out):,} rows out  ({n_removed} removed)")
    print(f"  Names corrected:        {n_name}")
    print(f"  Coordinates corrected:  {n_lat} lat  /  {n_lon} lon")
    print(f"  Place types corrected:  {n_type}")
   


✅ Updated  ../data/JMU/JMU_geoparsed_long.csv
✅ Saved    ../data/JMU/JMU_geoparsed_cleaned.csv

Summary:
  884 rows in  →  884 rows out  (0 removed)
  Names corrected:        131
  Coordinates corrected:  131 lat  /  131 lon
  Place types corrected:  131


---

## 8 Check Your Progress

Run the cell below at any time to see how many locations your team has reviewed, which checks pass, and your current grade estimate.


In [ ]:
%run ./tests/progress.py


---

## Lesson Summary

### Part 1: Load the Review File
- `pd.read_csv('file.csv')` — loads the raw geoparsed output for inspection before review

### Part 2: Setup Google Sheet
- Export the DataFrame to CSV, import it into Google Sheets, add data-validation dropdowns

### Part 3: Publish the Google Sheet
- Publish the sheet to a live `.csv` link that updates the data

### Part 4: Branch, Merge, Sync
- Branch, add URL to notebook, PR, and pull down changes

### Part 5: Recursive Review
- All team members review the data and update the Google Sheet

### Part 6: Data and Map
- Re-run the mapping cells periodically to check your progress

### Part 7: Export the Cleaned File
- Once data cleaning is complete, branch, export the cleaned file, PR and sync.

### Part 8: 

➡️ **Next:** [Lesson 5.1 — Sentiment Analysis](../lesson_5_sentiment_analysis/lesson_5_1_sentiment_analysis.ipynb)